# 第五天：大语言模型（LLM）在 MOF 智能设计中的应用

## 完整实操教程

本notebook涵盖：
1. 文本到结构生成（Text-to-Structure）
2. 文献挖掘与知识抽取
3. AutoML自动化模型优化
4. 智能体自主MOF发现
5. RAG系统构建
6. 实战案例：端到端MOF设计流程

---

## Part 1: 环境准备与导入

### 1.1 安装依赖

In [ ]:
# 基础依赖（应该已安装）
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from typing import Dict, List

# 检查导入
print("✓ 基础库导入成功")

# 导入我们的LLM工具
import sys
sys.path.append('../src')

from llm_tools.text_to_structure import Text2StructureGenerator, StructureValidator
from llm_tools.mof_agent_system import LiteratureMiner, AutoMLOptimizer, MOFAgent

print("✓ LLM工具导入成功")

### 1.2 配置（可选：使用真实LLM API）

如果您有OpenAI或Anthropic的API密钥，可以取消注释以下代码：

In [ ]:
# 使用MockLLM（演示）
USE_REAL_LLM = False

# 如果使用真实LLM，配置API密钥
if USE_REAL_LLM:
    # 方法1：使用OpenAI
    # from openai import OpenAI
    # client = OpenAI(api_key="your-api-key")
    
    # 方法2：使用Anthropic Claude
    # from anthropic import Anthropic
    # client = Anthropic(api_key="your-api-key")
    
    print("✓ 使用真实LLM API")
else:
    print("ℹ 使用MockLLM（演示模式）")
    print("  提示：要使用真实LLM，设置 USE_REAL_LLM=True 并配置API密钥")

---

## Part 2: 文本到结构生成（Text-to-Structure）

### 2.1 基础使用

In [ ]:
# 创建生成器
generator = Text2StructureGenerator()

# 示例查询
query = "Design a copper-based MOF with paddle-wheel clusters for CO2 capture"

print(f"查询: {query}\n")
print("="*70)

# 生成MOF候选
result = generator.generate_from_text(
    query,
    method="hybrid",  # retrieval | generation | hybrid
    top_k=3,
    verbose=True
)

### 2.2 查看和分析结果

In [ ]:
# 提取的参数
print("\n📋 提取的设计参数:")
params = result['extracted_parameters']
for key, value in params.items():
    print(f"  {key}: {value}")

# 候选MOF
print("\n🏆 Top 3 候选MOF:")
for i, cand in enumerate(result['candidates'][:3]):
    print(f"\n{i+1}. {cand['name']} (Score: {cand['score']:.2f})")
    print(f"   金属中心: {cand['metal']}")
    print(f"   有机配体: {cand['linker']}")
    print(f"   拓扑结构: {cand.get('topology', 'Unknown')}")
    print(f"   表面积: {cand.get('surface_area', 'N/A')} m²/g")
    print(f"   预测CO₂吸附: {cand['predicted_co2']:.2f} mmol/g")
    print(f"   来源: {cand['source']}")

### 2.3 验证和可视化

In [ ]:
# 验证top候选
top_candidate = result['candidates'][0]
is_valid, issues = StructureValidator.validate_structure(top_candidate)

print(f"\n✓ 最佳候选验证:")
print(f"  名称: {top_candidate['name']}")
print(f"  有效性: {'✓ 通过' if is_valid else '✗ 失败'}")
if issues:
    print(f"  问题: {', '.join(issues)}")

# 可视化候选MOF的性质对比
plt.figure(figsize=(10, 5))

# 子图1：CO2吸附量对比
plt.subplot(1, 2, 1)
names = [c['name'][:15] for c in result['candidates'][:5]]
co2_values = [c['predicted_co2'] for c in result['candidates'][:5]]
colors = ['#2ecc71' if c['source'] == 'database' else '#3498db' 
          for c in result['candidates'][:5]]

plt.barh(names, co2_values, color=colors)
plt.xlabel('CO₂ Uptake (mmol/g)')
plt.title('候选MOF的CO₂吸附量对比')
plt.legend(['Database', 'Generated'], loc='lower right')

# 子图2：综合评分
plt.subplot(1, 2, 2)
scores = [c['score'] for c in result['candidates'][:5]]
plt.barh(names, scores, color='#e74c3c')
plt.xlabel('综合评分')
plt.title('候选MOF的综合评分')

plt.tight_layout()
plt.show()

### 2.4 批量查询和对比

In [ ]:
# 定义多个查询
queries = [
    "High surface area Zr-MOF for CO2 storage",
    "Water-stable MOF with open metal sites",
    "MOF for selective CH4/CO2 separation",
]

# 批量处理
batch_results = []
for query in queries:
    result = generator.generate_from_text(query, method="hybrid", top_k=1, verbose=False)
    batch_results.append({
        'query': query,
        'best_mof': result['candidates'][0]
    })

# 创建对比表
comparison_df = pd.DataFrame([
    {
        '查询': r['query'][:40] + '...',
        '推荐MOF': r['best_mof']['name'],
        '金属': r['best_mof']['metal'],
        'CO₂吸附': f"{r['best_mof']['predicted_co2']:.2f}",
        '评分': f"{r['best_mof']['score']:.2f}"
    }
    for r in batch_results
])

print("\n📊 批量查询结果对比:")
print(comparison_df.to_string(index=False))

---

## Part 3: 文献挖掘与知识抽取

### 3.1 从单篇论文提取信息

In [ ]:
# 创建文献挖掘器
miner = LiteratureMiner()

# 模拟论文摘要/正文
paper_text = """
UiO-66 is a zirconium-based metal-organic framework synthesized from
ZrCl4 and 1,4-benzenedicarboxylic acid (H2BDC) in dimethylformamide (DMF)
at 120°C for 24 hours. The resulting material exhibits a BET surface area
of 1200 m²/g and shows excellent water stability. CO2 adsorption measurements
at 298K and 1 bar revealed an uptake of 3.0 mmol/g. The framework adopts
the fcu topology with Zr6O4(OH)4 clusters as secondary building units.
"""

print("📄 分析论文...\n")
extracted_info = miner.mine_paper(paper_text)

# 显示提取的信息
print("✓ 提取完成！\n")
print(json.dumps(extracted_info, indent=2, ensure_ascii=False))

### 3.2 构建MOF知识图谱

In [ ]:
# 模拟多篇论文
papers = [
    """
    MOF-5 was synthesized using Zn(NO3)2 and 1,4-benzenedicarboxylic acid (H2BDC)
    in diethylformamide at 100°C. The cubic framework shows a surface area of
    3800 m²/g and CO2 uptake of 4.5 mmol/g at room temperature.
    """,
    """
    HKUST-1 (also known as Cu-BTC) contains copper paddle-wheel units connected
    by 1,3,5-benzenetricarboxylate (BTC) linkers. Synthesized in water/ethanol
    at 120°C, it exhibits 1850 m²/g surface area and 6.5 mmol/g CO2 uptake.
    The material features open metal sites beneficial for gas adsorption.
    """,
    """
    UiO-66-NH2 is a functionalized variant of UiO-66 using 2-amino-BDC linker.
    Zr-based framework synthesized in DMF at 120°C shows 1100 m²/g surface area.
    The -NH2 groups enhance CO2 affinity, achieving 3.5 mmol/g uptake.
    """
]

# 构建知识图谱
print("🔨 构建知识图谱...\n")
knowledge_graph = miner.build_knowledge_graph(papers)

# 显示知识图谱
print(f"✓ 知识图谱包含 {len(knowledge_graph)} 个MOF实体\n")

for mof_name, entries in knowledge_graph.items():
    print(f"\n{'='*60}")
    print(f"MOF: {mof_name}")
    print(f"{'='*60}")
    
    for i, entry in enumerate(entries, 1):
        print(f"\n  条目 #{i}:")
        print(f"    金属: {', '.join(entry['metals']) if entry['metals'] else 'Unknown'}")
        print(f"    配体: {', '.join(entry['linkers']) if entry['linkers'] else 'Unknown'}")
        print(f"    合成条件:")
        synth = entry['synthesis_conditions']
        print(f"      温度: {synth.get('temperature', 'N/A')}°C")
        print(f"      溶剂: {', '.join(synth.get('solvents', ['N/A']))}")
        print(f"    性质:")
        props = entry['properties']
        if 'surface_area' in props:
            print(f"      表面积: {props['surface_area']} m²/g")
        if 'co2_uptake' in props:
            print(f"      CO₂吸附: {props['co2_uptake']} mmol/g")

### 3.3 语义搜索

In [ ]:
# 执行语义搜索
search_queries = [
    "Zr-based MOF for CO2 capture",
    "Copper MOF with high uptake",
    "Functionalized framework"
]

print("🔍 语义搜索示例:\n")
for query in search_queries:
    results = miner.semantic_search(query, top_k=3)
    print(f"查询: '{query}'")
    print(f"  结果: {', '.join(results) if results else '无匹配'}\n")

---

## Part 4: AutoML 自动化模型优化

### 4.1 准备数据

In [ ]:
# 生成模拟MOF数据集
np.random.seed(42)

n_samples = 200

# 特征：表面积、孔体积、金属电负性、配体极化率等
surface_area = np.random.uniform(500, 5000, n_samples)
pore_volume = np.random.uniform(0.2, 2.0, n_samples)
metal_electronegativity = np.random.uniform(1.0, 2.5, n_samples)
linker_length = np.random.uniform(5, 20, n_samples)
temperature = np.random.uniform(77, 373, n_samples)

# 组合特征矩阵
X = np.column_stack([
    surface_area,
    pore_volume,
    metal_electronegativity,
    linker_length,
    temperature
])

# 目标：CO2吸附量（基于物理规律的模拟）
y = (
    0.0015 * surface_area + 
    1.5 * pore_volume + 
    0.8 * metal_electronegativity +
    0.1 * linker_length -
    0.01 * temperature +
    np.random.normal(0, 0.5, n_samples)
)

# 创建DataFrame
feature_names = ['surface_area', 'pore_volume', 'metal_electronegativity', 
                 'linker_length', 'temperature']
df = pd.DataFrame(X, columns=feature_names)
df['co2_uptake'] = y

print("📊 MOF数据集统计:\n")
print(df.describe())

# 可视化数据分布
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for i, col in enumerate(feature_names + ['co2_uptake']):
    axes[i].hist(df[col], bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

### 4.2 运行AutoML优化

In [ ]:
# 创建AutoML优化器
optimizer = AutoMLOptimizer()

# 运行优化
print("\n🤖 开始AutoML优化...\n")
print("="*70)

result = optimizer.auto_optimize(
    X=X,
    y=y,
    task_description="Predict CO2 uptake from MOF structural features",
    max_iterations=8
)

print("\n" + "="*70)
print("✓ 优化完成！\n")
print(f"最佳模型: {result['best_model']['name']}")
print(f"最佳R²评分: {result['best_score']:.4f}")
print(f"\n最佳超参数:")
for key, value in result['best_model']['params'].items():
    print(f"  {key}: {value}")

### 4.3 可视化优化过程

In [ ]:
# 提取优化历史
history = result['history']

# 创建可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 子图1：优化曲线
iterations = [h['iteration'] for h in history]
scores = [h['score'] for h in history]

axes[0].plot(iterations, scores, marker='o', linewidth=2, markersize=8)
axes[0].axhline(y=result['best_score'], color='r', linestyle='--', 
                label=f"Best Score: {result['best_score']:.4f}")
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('R² Score', fontsize=12)
axes[0].set_title('AutoML Optimization Progress', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 子图2：不同模型的性能对比
model_scores = {}
for h in history:
    model_name = h['model']
    if model_name not in model_scores:
        model_scores[model_name] = []
    model_scores[model_name].append(h['score'])

model_avg_scores = {k: np.mean(v) for k, v in model_scores.items()}

models = list(model_avg_scores.keys())
avg_scores = list(model_avg_scores.values())
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

axes[1].barh(models, avg_scores, color=colors[:len(models)])
axes[1].set_xlabel('Average R² Score', fontsize=12)
axes[1].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[1].grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📈 模型性能排名:")
sorted_models = sorted(model_avg_scores.items(), key=lambda x: x[1], reverse=True)
for i, (model, score) in enumerate(sorted_models, 1):
    print(f"{i}. {model:15s} - R² = {score:.4f}")

---

## Part 5: MOF智能体自主发现

### 5.1 创建并运行智能体

In [ ]:
# 创建MOF发现智能体
agent = MOFAgent()

# 定义研究目标
goal = "Discover a MOF with CO2 uptake greater than 5.5 mmol/g"

print("🤖 启动MOF发现智能体\n")
print("="*70)

# 运行自主发现
report = agent.discover_mof(
    goal=goal,
    max_iterations=5,
    verbose=True
)

print("\n" + "="*70)
print("✓ 发现任务完成！")

### 5.2 分析发现报告

In [ ]:
print("\n" + "="*70)
print(" "*25 + "MOF 发现报告")
print("="*70)

print(f"\n📌 研究目标: {report['goal']}")
print(f"\n🎯 任务状态: {'✓ 成功达成' if report['success'] else '✗ 未达成'}")
print(f"\n🔄 迭代次数: {report['iterations']}")

print(f"\n🏆 最佳发现:")
best_mof = report['best_mof']
print(f"  名称: {best_mof['name']}")
print(f"  预测CO₂吸附: {best_mof['predicted_value']:.2f} mmol/g")
if best_mof.get('base_mof'):
    print(f"  基于: {best_mof['base_mof']}")
print(f"  修饰策略: {best_mof['modifications']}")

print(f"\n📊 所有发现的MOF:")
for i, mof in enumerate(report['discovered_mofs'], 1):
    status = "✓" if mof['predicted_value'] >= report['target_value'] else "○"
    print(f"{status} {i}. {mof['name']:20s} - {mof['predicted_value']:.2f} mmol/g")

print(f"\n📈 学习曲线: {[f'{v:.2f}' for v in report['learning_curve']]}")

### 5.3 可视化发现过程

In [ ]:
# 可视化学习曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 子图1：迭代性能提升
iterations = list(range(1, len(report['learning_curve']) + 1))
values = report['learning_curve']
target = report['target_value']

axes[0].plot(iterations, values, marker='o', linewidth=2.5, 
             markersize=10, color='#3498db', label='Predicted Value')
axes[0].axhline(y=target, color='#e74c3c', linestyle='--', 
                linewidth=2, label=f'Target: {target} mmol/g')
axes[0].fill_between(iterations, target, values, 
                      where=[v >= target for v in values],
                      color='green', alpha=0.2, label='Goal Achieved')
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('CO₂ Uptake (mmol/g)', fontsize=12)
axes[0].set_title('Agent Learning Progress', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 子图2：所有候选MOF分布
mof_names = [m['name'][:15] for m in report['discovered_mofs']]
mof_values = [m['predicted_value'] for m in report['discovered_mofs']]
colors_bar = ['#2ecc71' if v >= target else '#95a5a6' for v in mof_values]

axes[1].barh(mof_names, mof_values, color=colors_bar)
axes[1].axvline(x=target, color='#e74c3c', linestyle='--', linewidth=2)
axes[1].set_xlabel('CO₂ Uptake (mmol/g)', fontsize=12)
axes[1].set_title('All Discovered MOFs', fontsize=14, fontweight='bold')
axes[1].grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

### 5.4 智能体学习分析

In [ ]:
# 让智能体从经验中学习
print("\n🧠 智能体经验学习:\n")
print("="*70)

agent.learn_from_experience()

print("\n" + "="*70)

---

## Part 6: 综合案例 - 端到端MOF设计流程

### 6.1 定义设计需求

In [ ]:
# 完整的MOF设计流程
class CompleteMOFDesignPipeline:
    """
    完整的端到端MOF设计流程
    
    流程:
    1. 需求分析（文本理解）
    2. 文献调研（知识提取）
    3. 候选生成（Text-to-Structure）
    4. 性质预测（AutoML）
    5. 优化发现（Agent）
    6. 报告生成
    """
    
    def __init__(self):
        self.text2struct = Text2StructureGenerator()
        self.miner = LiteratureMiner()
        self.automl = AutoMLOptimizer()
        self.agent = MOFAgent()
        
    def design(self, requirements: str, reference_papers: List[str] = None):
        print("\n" + "="*70)
        print(" "*20 + "MOF智能设计系统")
        print("="*70)
        
        print(f"\n📝 设计需求:\n{requirements}\n")
        
        # Step 1: 文献调研
        print("\n" + "-"*70)
        print("[Step 1/5] 文献调研与知识提取")
        print("-"*70)
        
        if reference_papers:
            kg = self.miner.build_knowledge_graph(reference_papers)
            print(f"✓ 从{len(reference_papers)}篇论文中提取了{len(kg)}个MOF实体")
        else:
            print("⊘ 未提供参考文献，跳过")
        
        # Step 2: 初始候选生成
        print("\n" + "-"*70)
        print("[Step 2/5] 生成初始MOF候选")
        print("-"*70)
        
        initial_result = self.text2struct.generate_from_text(
            requirements,
            method="hybrid",
            top_k=5,
            verbose=False
        )
        print(f"✓ 生成了{len(initial_result['candidates'])}个候选MOF")
        
        # Step 3: 性质预测模型训练
        print("\n" + "-"*70)
        print("[Step 3/5] AutoML性质预测模型训练")
        print("-"*70)
        print("ℹ 使用模拟数据训练（实际应用中应使用真实MOF数据）")
        
        # 模拟训练过程（简化）
        print("✓ 模型训练完成（模拟）")
        
        # Step 4: 智能体优化
        print("\n" + "-"*70)
        print("[Step 4/5] 智能体自主优化")
        print("-"*70)
        
        # 提取目标
        goal = self._extract_goal(requirements)
        print(f"提取的优化目标: {goal}")
        
        agent_report = self.agent.discover_mof(
            goal=goal,
            max_iterations=3,
            verbose=False
        )
        print(f"✓ 智能体完成{agent_report['iterations']}次迭代")
        
        # Step 5: 生成最终报告
        print("\n" + "-"*70)
        print("[Step 5/5] 生成设计报告")
        print("-"*70)
        
        final_report = self._generate_final_report(
            requirements,
            initial_result,
            agent_report
        )
        
        return final_report
    
    def _extract_goal(self, requirements: str) -> str:
        # 简化的目标提取
        if "co2" in requirements.lower() or "carbon dioxide" in requirements.lower():
            return "Maximize CO2 uptake"
        elif "surface area" in requirements.lower():
            return "Maximize surface area"
        else:
            return "General MOF optimization"
    
    def _generate_final_report(self, requirements, initial_result, agent_report):
        report = {
            'requirements': requirements,
            'initial_candidates': initial_result['candidates'][:3],
            'optimized_mof': agent_report['best_mof'],
            'agent_iterations': agent_report['iterations'],
            'success': agent_report['success']
        }
        
        # 打印报告
        print("\n" + "="*70)
        print(" "*25 + "最终设计报告")
        print("="*70)
        
        print(f"\n✓ 设计流程完成！")
        print(f"\n推荐MOF: {report['optimized_mof']['name']}")
        print(f"预测性能: {report['optimized_mof']['predicted_value']:.2f} mmol/g")
        print(f"\n初始候选数: {len(report['initial_candidates'])}")
        print(f"优化迭代数: {report['agent_iterations']}")
        print(f"任务状态: {'成功' if report['success'] else '需继续优化'}")
        
        return report

# 创建设计流程
pipeline = CompleteMOFDesignPipeline()

print("✓ 端到端设计流程已准备就绪")

### 6.2 执行完整设计流程

In [ ]:
# 定义设计需求
design_requirements = """
设计目标：开发用于工业烟气CO2捕获的MOF材料

性能要求：
- CO2吸附量 > 5 mmol/g (298K, 1 bar)
- CO2/N2选择性 > 50
- 水稳定性良好
- 可规模化合成

优先考虑：
- 使用成本较低的金属中心（Zr, Al, 或Cu）
- 商业化配体或易合成配体
- 已有成功合成先例的拓扑结构
"""

# 模拟参考文献
reference_papers = [
    """
    UiO-66 demonstrates excellent water stability due to strong Zr-O bonds.
    Synthesized from ZrCl4 and H2BDC in DMF at 120°C. Surface area: 1200 m²/g,
    CO2 uptake: 3.0 mmol/g.
    """,
    """
    HKUST-1 shows high CO2 uptake (6.5 mmol/g) with open Cu sites.
    Synthesized from Cu(NO3)2 and H3BTC. However, sensitive to moisture.
    """
]

# 运行设计流程
final_report = pipeline.design(
    requirements=design_requirements,
    reference_papers=reference_papers
)

---

## Part 7: 总结与扩展

### 7.1 本教程涵盖内容

In [ ]:
summary = """
╔════════════════════════════════════════════════════════════════════╗
║          第五天：LLM在MOF智能设计中的应用 - 教程总结                ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  ✓ Part 1: 环境准备                                                ║
║    - MockLLM演示模式                                                ║
║    - 真实LLM API集成准备                                            ║
║                                                                    ║
║  ✓ Part 2: 文本到结构生成                                           ║
║    - 3种生成方法（检索、生成、混合）                                  ║
║    - 批量查询和结果对比                                              ║
║    - 结构验证和可视化                                                ║
║                                                                    ║
║  ✓ Part 3: 文献挖掘                                                 ║
║    - 信息提取（MOF名称、金属、配体、性质）                            ║
║    - 知识图谱构建                                                    ║
║    - 语义搜索                                                        ║
║                                                                    ║
║  ✓ Part 4: AutoML优化                                               ║
║    - 自动特征工程                                                    ║
║    - 模型选择和超参数优化                                            ║
║    - 优化过程可视化                                                  ║
║                                                                    ║
║  ✓ Part 5: MOF智能体                                                ║
║    - 自主发现循环                                                    ║
║    - 经验学习                                                        ║
║    - 发现过程可视化                                                  ║
║                                                                    ║
║  ✓ Part 6: 端到端设计流程                                           ║
║    - 需求→文献→候选→预测→优化→报告                                   ║
║    - 完整的工业级设计流程                                            ║
║                                                                    ║
║  📚 学习成果：                                                       ║
║    - 掌握LLM在材料设计中的应用                                        ║
║    - 理解RAG、AutoML、Agent等前沿技术                                 ║
║    - 能够构建完整的智能设计系统                                       ║
║                                                                    ║
╚════════════════════════════════════════════════════════════════════╝
"""

print(summary)

### 7.2 进一步扩展方向

In [ ]:
extensions = """
🚀 扩展方向建议:

1. 集成真实LLM API
   - OpenAI GPT-4: 强大但成本高
   - Anthropic Claude: 长上下文能力
   - 本地LLaMA: 隐私和成本优势

2. 构建专业知识库
   - 收集MOF文献PDF
   - 使用LlamaIndex构建向量数据库
   - 实现高质量RAG系统

3. 增强AutoML功能
   - 集成更多ML库（AutoGluon, TPOT）
   - 神经架构搜索（NAS）
   - 超参数优化（Optuna, Ray Tune）

4. 多智能体系统
   - 文献专家、设计师、预测师协作
   - 强化学习智能体
   - 人机协作界面（Gradio/Streamlit）

5. 实验室自动化集成
   - LLM生成实验方案
   - 机器人执行合成
   - 闭环优化系统

6. 模型微调
   - 收集MOF领域数据
   - 微调LLaMA/Mistral
   - 创建MOF专用模型
"""

print(extensions)

### 7.3 练习题

In [ ]:
exercises = """
📝 练习题:

基础练习:
1. 修改文本查询，生成针对CH4存储的MOF
2. 从3篇新论文构建知识图谱
3. 用AutoML预测其他性质（如选择性、稳定性）

进阶练习:
4. 实现多目标优化智能体（同时优化uptake和稳定性）
5. 添加可合成性评估模块
6. 构建交互式Gradio界面

挑战练习:
7. 集成真实OpenAI API，对比与MockLLM的差异
8. 使用真实MOF数据集（QMOF/CoRE-MOF）训练AutoML
9. 实现多智能体协作系统（3个以上agent）
10. 构建完整的MOF设计→合成→测试闭环系统
"""

print(exercises)

---

## 恭喜！🎉

您已完成第五天的完整教程。现在您已经掌握了：

- ✅ LLM在材料科学中的应用
- ✅ 文本到结构生成技术
- ✅ 自动化文献挖掘
- ✅ AutoML模型优化
- ✅ 自主学习智能体
- ✅ 端到端MOF设计流程

### 下一步建议：

1. **实践应用**：在真实MOF数据上运行这些工具
2. **深入学习**：阅读最新的LLM+材料论文
3. **贡献开源**：改进代码，分享给社区
4. **发表研究**：将LLM辅助的发现写成论文

祝研究顺利！🚀